# Sonar Signal — Unsupervised Machine Learning Project
**Dataset:** 208 sonar returns × 60 frequency-band energy features  
**Goal:** Discover hidden structure without labels using clustering, dimensionality reduction & anomaly detection.

---
## Table of Contents
1. Imports & Setup  
2. Load & Inspect Data  
3. Exploratory Data Analysis (EDA)  
4. Pre-processing  
5. Dimensionality Reduction — PCA  
6. Dimensionality Reduction — t-SNE  
7. K-Means Clustering (Elbow + Silhouette)  
8. Hierarchical / Agglomerative Clustering  
9. DBSCAN  
10. Gaussian Mixture Model (GMM)  
11. Anomaly Detection — Isolation Forest  
12. Anomaly Detection — Local Outlier Factor  
13. Cluster Profiling & Interpretation  
14. Summary

## 1. Imports & Setup

In [ ]:
#  Standard libraries 
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
sns.set_style('whitegrid')

#  Pre-processing 
from sklearn.preprocessing import StandardScaler

#  Dimensionality reduction 
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

#  Clustering 
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture

#  Evaluation metrics 
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

#  Anomaly detection 
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors

#  Hierarchical linkage / dendrogram 
from scipy.cluster.hierarchy import dendrogram, linkage

print('All imports successful')

## 2. Load & Inspect Data

In [ ]:
# The dataset has NO header row and NO label column (purely unsupervised)
df = pd.read_csv('sonar_unsupervised_data.csv', header=None)

# Name columns F1 ... F60
df.columns = [f'F{i+1}' for i in range(df.shape[1])]

print(f'Shape : {df.shape}   ->  208 samples, 60 frequency-band features')
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Data types:')
print(df.dtypes.value_counts())
print('\nMissing values:', df.isnull().sum().sum())
print('\nBasic statistics:')
df.describe().T.round(4)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
#  3a. Distribution of 6 representative features 
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sample_feats = ['F1', 'F10', 'F20', 'F30', 'F50', 'F60']
for ax, col in zip(axes.flatten(), sample_feats):
    ax.hist(df[col], bins=20, color='steelblue', edgecolor='white')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.suptitle('Histogram of Selected Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
#  3b. Box-plots across all 60 features 
plt.figure(figsize=(18, 5))
df.boxplot(ax=plt.gca(), rot=90, grid=False,
           medianprops=dict(color='red', linewidth=2))
plt.title('Box-Plot of All 60 Features')
plt.xlabel('Feature')
plt.ylabel('Value')
plt.tight_layout()
plt.show()

In [ ]:
#  3c. Correlation heat-map (first 20 features) 
plt.figure(figsize=(14, 10))
corr = df.iloc[:, :20].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm',
            linewidths=0.3, vmin=-1, vmax=1)
plt.title('Correlation Heat-map (Features F1 - F20)')
plt.tight_layout()
plt.show()

In [ ]:
#  3d. Mean energy profile across frequency bands 
mean_profile = df.mean()
plt.figure(figsize=(14, 4))
plt.plot(range(1, 61), mean_profile.values, marker='o', markersize=3, color='darkorange')
plt.fill_between(range(1, 61), mean_profile.values, alpha=0.3, color='darkorange')
plt.title('Average Energy Profile Across 60 Frequency Bands')
plt.xlabel('Frequency Band Index')
plt.ylabel('Mean Energy')
plt.tight_layout()
plt.show()

## 4. Pre-processing

In [ ]:
# StandardScaler: zero mean, unit variance
# Essential before any distance-based algorithm
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print(f'Scaled array shape : {X_scaled.shape}')
print(f'Mean (should ~0)   : {X_scaled.mean():.6f}')
print(f'Std  (should ~1)   : {X_scaled.std():.6f}')

## 5. Dimensionality Reduction — PCA

In [ ]:
pca_full = PCA(n_components=60, random_state=42)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95   = int(np.argmax(cumvar >= 0.95)) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 61), pca_full.explained_variance_ratio_,
            color='steelblue', edgecolor='white')
axes[0].set_title('Explained Variance per Component')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')

axes[1].plot(range(1, 61), cumvar, marker='.', color='darkorange')
axes[1].axhline(0.95, linestyle='--', color='red', label='95% threshold')
axes[1].axvline(n_95, linestyle='--', color='green', label=f'{n_95} components')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'\n-> {n_95} principal components explain >= 95% of the variance.')

In [ ]:
# 2-D PCA for visualisation
pca_2d = PCA(n_components=2, random_state=42)
X_pca2 = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca2[:, 0], X_pca2[:, 1], alpha=0.6, s=40,
            color='steelblue', edgecolors='white')
plt.title('PCA - 2D Projection of Sonar Data')
plt.xlabel(f'PC1  ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2  ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.tight_layout()
plt.show()

In [ ]:
# Reduced matrix for modelling (95% variance)
pca_95 = PCA(n_components=n_95, random_state=42)
X_pca  = pca_95.fit_transform(X_scaled)
print(f'Reduced feature matrix shape: {X_pca.shape}')

## 6. Dimensionality Reduction — t-SNE

In [ ]:
tsne   = TSNE(n_components=2, perplexity=30, learning_rate=200,
              n_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X_pca)   # input PCA-reduced data for speed

plt.figure(figsize=(8, 6))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], alpha=0.7, s=50,
            color='mediumpurple', edgecolors='white')
plt.title('t-SNE 2D Projection of Sonar Data')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.tight_layout()
plt.show()

## 7. K-Means Clustering

In [ ]:
#  Elbow Method 
K_range    = range(2, 11)
inertia_vals = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_pca)
    inertia_vals.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(list(K_range), inertia_vals, marker='o', color='steelblue', linewidth=2)
plt.title('Elbow Method - Optimal Number of Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Within-Cluster SSE)')
plt.xticks(list(K_range))
plt.tight_layout()
plt.show()

In [ ]:
#  Silhouette Score 
sil_scores = []
for k in K_range:
    km  = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    lbl = km.fit_predict(X_pca)
    sil_scores.append(silhouette_score(X_pca, lbl))

plt.figure(figsize=(9, 5))
plt.plot(list(K_range), sil_scores, marker='s', color='darkorange', linewidth=2)
plt.title('Silhouette Score vs. Number of Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score  (higher = better)')
plt.xticks(list(K_range))
plt.tight_layout()
plt.show()

best_k = list(K_range)[int(np.argmax(sil_scores))]
print(f'Best k by Silhouette = {best_k}  (score = {max(sil_scores):.4f})')

In [ ]:
#  Final K-Means 
kmeans        = KMeans(n_clusters=best_k, init='k-means++', n_init=10, random_state=42)
kmeans_labels = kmeans.fit_predict(X_pca)
df['KMeans_Cluster'] = kmeans_labels

print('Cluster distribution:')
print(pd.Series(kmeans_labels).value_counts().sort_index())
print(f'\nSilhouette Score  : {silhouette_score(X_pca, kmeans_labels):.4f}')
print(f'Davies-Bouldin    : {davies_bouldin_score(X_pca, kmeans_labels):.4f}  (lower = better)')
print(f'Calinski-Harabasz : {calinski_harabasz_score(X_pca, kmeans_labels):.4f}  (higher = better)')

In [ ]:
#  K-Means on PCA 2D 
palette = sns.color_palette('tab10', best_k)
plt.figure(figsize=(8, 6))
for c in range(best_k):
    mask = kmeans_labels == c
    plt.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                label=f'Cluster {c}', s=50, alpha=0.8,
                color=palette[c], edgecolors='white')

# Project cluster centres back to 2D PCA space
centres_2d = pca_2d.transform(pca_95.inverse_transform(kmeans.cluster_centers_))
plt.scatter(centres_2d[:, 0], centres_2d[:, 1],
            c='black', marker='X', s=150, zorder=5, label='Centroids')
plt.title(f'K-Means Clusters (k={best_k}) - PCA 2D')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
#  K-Means on t-SNE 
plt.figure(figsize=(8, 6))
for c in range(best_k):
    mask = kmeans_labels == c
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                label=f'Cluster {c}', s=50, alpha=0.8,
                color=palette[c], edgecolors='white')
plt.title(f'K-Means Clusters (k={best_k}) - t-SNE')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.legend(); plt.tight_layout(); plt.show()

## 8. Hierarchical / Agglomerative Clustering

In [ ]:
#  Dendrogram (50-sample subset for readability) 
rng        = np.random.RandomState(42)
sample_idx = rng.choice(len(X_pca), 50, replace=False)
linked     = linkage(X_pca[sample_idx], method='ward')

plt.figure(figsize=(16, 5))
dendrogram(linked, leaf_rotation=90, leaf_font_size=7,
           color_threshold=0.7 * max(linked[:, 2]))
plt.title('Hierarchical Clustering Dendrogram (50-sample subset, Ward linkage)')
plt.xlabel('Sample Index')
plt.ylabel('Euclidean Distance')
plt.tight_layout()
plt.show()

In [ ]:
#  Agglomerative Clustering on full data 
agg        = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
agg_labels = agg.fit_predict(X_pca)
df['Agglomerative_Cluster'] = agg_labels

print('Cluster distribution:')
print(pd.Series(agg_labels).value_counts().sort_index())
print(f'\nSilhouette Score  : {silhouette_score(X_pca, agg_labels):.4f}')
print(f'Davies-Bouldin    : {davies_bouldin_score(X_pca, agg_labels):.4f}')

plt.figure(figsize=(8, 6))
for c in range(best_k):
    mask = agg_labels == c
    plt.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                label=f'Cluster {c}', s=50, alpha=0.8,
                color=palette[c], edgecolors='white')
plt.title(f'Agglomerative Clusters (k={best_k}) - PCA 2D')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(); plt.tight_layout(); plt.show()

## 9. DBSCAN

In [ ]:
#  k-distance plot to help choose eps 
k_nn = 5
nn   = NearestNeighbors(n_neighbors=k_nn)
nn.fit(X_pca)
distances, _ = nn.kneighbors(X_pca)
k_dist       = np.sort(distances[:, -1])[::-1]

plt.figure(figsize=(10, 4))
plt.plot(k_dist, color='teal')
plt.title(f'k-Distance Plot (k={k_nn}) - Elbow suggests eps')
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{k_nn}-th Nearest Neighbour Distance')
plt.tight_layout()
plt.show()

In [ ]:
#  Fit DBSCAN 
eps_val  = float(np.percentile(k_dist, 10))  # automated elbow approximation
min_samp = max(5, 2 * X_pca.shape[1] // 10)

db       = DBSCAN(eps=eps_val, min_samples=min_samp)
db_labels = db.fit_predict(X_pca)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise       = list(db_labels).count(-1)
print(f'eps = {eps_val:.4f}  |  min_samples = {min_samp}')
print(f'Clusters found : {n_clusters_db}')
print(f'Noise points   : {n_noise}')

non_noise = db_labels != -1
if n_clusters_db > 1 and non_noise.sum() > 1:
    print(f'Silhouette (non-noise) : {silhouette_score(X_pca[non_noise], db_labels[non_noise]):.4f}')

# Visualise
unique_lbls = sorted(set(db_labels))
n_real_cl   = max(1, n_clusters_db)
col_pal     = sns.color_palette('tab10', n_real_cl)
col_map     = {}
ci = 0
for lbl in unique_lbls:
    if lbl == -1:
        col_map[lbl] = 'black'
    else:
        col_map[lbl] = col_pal[ci % n_real_cl]
        ci += 1

plt.figure(figsize=(8, 6))
for lbl in unique_lbls:
    mask = db_labels == lbl
    name = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    plt.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                label=name, s=50, alpha=0.7,
                color=col_map[lbl], edgecolors='white')
plt.title('DBSCAN Clusters - PCA 2D')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(); plt.tight_layout(); plt.show()

## 10. Gaussian Mixture Model (GMM)

In [ ]:
#  BIC / AIC to pick number of components 
bic_scores = []
aic_scores = []

for k in K_range:
    gmm = GaussianMixture(n_components=k, covariance_type='full',
                          random_state=42, n_init=3)
    gmm.fit(X_pca)
    bic_scores.append(gmm.bic(X_pca))
    aic_scores.append(gmm.aic(X_pca))

plt.figure(figsize=(10, 5))
plt.plot(list(K_range), bic_scores, marker='o', label='BIC', color='steelblue')
plt.plot(list(K_range), aic_scores, marker='s', label='AIC', color='darkorange')
plt.title('GMM - BIC & AIC vs. Number of Components')
plt.xlabel('Number of Components')
plt.ylabel('Score (lower = better)')
plt.legend(); plt.tight_layout(); plt.show()

best_gmm_k = list(K_range)[int(np.argmin(bic_scores))]
print(f'Best GMM components by BIC = {best_gmm_k}')

In [ ]:
#  Fit final GMM 
gmm        = GaussianMixture(n_components=best_gmm_k, covariance_type='full',
                             random_state=42, n_init=5)
gmm.fit(X_pca)
gmm_labels = gmm.predict(X_pca)
df['GMM_Cluster'] = gmm_labels

print('Cluster distribution:')
print(pd.Series(gmm_labels).value_counts().sort_index())
print(f'\nSilhouette Score  : {silhouette_score(X_pca, gmm_labels):.4f}')

pal_gmm = sns.color_palette('tab10', best_gmm_k)
plt.figure(figsize=(8, 6))
for c in range(best_gmm_k):
    mask = gmm_labels == c
    plt.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                label=f'Component {c}', s=50, alpha=0.8,
                color=pal_gmm[c], edgecolors='white')
plt.title(f'GMM Clusters ({best_gmm_k} components) - PCA 2D')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(); plt.tight_layout(); plt.show()

## 11. Anomaly Detection — Isolation Forest

In [ ]:
iso      = IsolationForest(n_estimators=200, contamination='auto', random_state=42)
iso_pred = iso.fit_predict(X_scaled)       # 1 = normal, -1 = anomaly
iso_scores = iso.decision_function(X_scaled)  # more negative = more anomalous

n_anomalies = (iso_pred == -1).sum()
print(f'Anomalies detected: {n_anomalies} / {len(iso_pred)}')

plt.figure(figsize=(10, 4))
plt.hist(iso_scores, bins=30, color='steelblue', edgecolor='white')
threshold_line = iso_scores[iso_pred == -1].max()
plt.axvline(threshold_line, linestyle='--', color='red', label='Anomaly threshold')
plt.title('Isolation Forest - Anomaly Score Distribution')
plt.xlabel('Anomaly Score (more negative = more anomalous)')
plt.ylabel('Count')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
normal_mask  = iso_pred ==  1
anomaly_mask = iso_pred == -1
plt.scatter(X_pca2[normal_mask,  0], X_pca2[normal_mask,  1],
            label='Normal',  s=40, alpha=0.6, color='steelblue', edgecolors='white')
plt.scatter(X_pca2[anomaly_mask, 0], X_pca2[anomaly_mask, 1],
            label='Anomaly', s=80, alpha=0.9, color='red',
            marker='x', linewidths=1.5)
plt.title('Isolation Forest Anomalies - PCA 2D')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(); plt.tight_layout(); plt.show()

## 12. Anomaly Detection — Local Outlier Factor (LOF)

In [ ]:
lof        = LocalOutlierFactor(n_neighbors=20, contamination='auto')
lof_pred   = lof.fit_predict(X_scaled)     # 1 = inlier, -1 = outlier
lof_scores = -lof.negative_outlier_factor_ # higher = more anomalous

n_outliers = (lof_pred == -1).sum()
print(f'Outliers detected by LOF: {n_outliers} / {len(lof_pred)}')

plt.figure(figsize=(8, 6))
inlier_mask  = lof_pred ==  1
outlier_mask = lof_pred == -1
plt.scatter(X_pca2[inlier_mask,  0], X_pca2[inlier_mask,  1],
            label='Inlier',  s=40, alpha=0.6, color='teal', edgecolors='white')
plt.scatter(X_pca2[outlier_mask, 0], X_pca2[outlier_mask, 1],
            label='Outlier', s=100, alpha=0.9, color='crimson',
            marker='D', edgecolors='black')
plt.title('Local Outlier Factor (LOF) - PCA 2D')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(); plt.tight_layout(); plt.show()

## 13. Cluster Profiling & Interpretation

In [ ]:
feature_cols = [c for c in df.columns if c.startswith('F')]
profile      = df.groupby('KMeans_Cluster')[feature_cols].mean()

plt.figure(figsize=(18, max(3, best_k + 1)))
sns.heatmap(profile, cmap='YlOrRd', linewidths=0.2,
            yticklabels=[f'Cluster {i}' for i in profile.index])
plt.title('K-Means Cluster Profiles - Mean Feature Value per Cluster')
plt.xlabel('Feature (F1 to F60,  low to high frequency)')
plt.tight_layout()
plt.show()

In [ ]:
#  Radar chart (every 5th feature) 
radar_feats = [f'F{i}' for i in range(1, 61, 5)]
n_r         = len(radar_feats)
angles      = np.linspace(0, 2 * np.pi, n_r, endpoint=False).tolist()
angles     += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
pal_r   = sns.color_palette('tab10', best_k)

for c in range(best_k):
    vals  = profile.loc[c, radar_feats].values.tolist()
    vals += vals[:1]
    ax.plot(angles, vals, label=f'Cluster {c}', color=pal_r[c], linewidth=2)
    ax.fill(angles, vals, alpha=0.15, color=pal_r[c])

ax.set_thetagrids(np.degrees(angles[:-1]), radar_feats, fontsize=9)
ax.set_title('Cluster Mean Energy - Radar Chart', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

In [ ]:
#  Algorithm comparison table 
summary = pd.DataFrame({
    'Algorithm'             : ['K-Means', 'Agglomerative', 'GMM'],
    'n_clusters'            : [best_k, best_k, best_gmm_k],
    'Silhouette (higher better)' : [
        round(silhouette_score(X_pca, kmeans_labels), 4),
        round(silhouette_score(X_pca, agg_labels),    4),
        round(silhouette_score(X_pca, gmm_labels),    4),
    ],
    'Davies-Bouldin (lower better)': [
        round(davies_bouldin_score(X_pca, kmeans_labels), 4),
        round(davies_bouldin_score(X_pca, agg_labels),    4),
        round(davies_bouldin_score(X_pca, gmm_labels),    4),
    ],
    'Calinski-Harabasz (higher better)': [
        round(calinski_harabasz_score(X_pca, kmeans_labels), 2),
        round(calinski_harabasz_score(X_pca, agg_labels),    2),
        round(calinski_harabasz_score(X_pca, gmm_labels),    2),
    ],
})
print(summary.to_string(index=False))

## 14. Summary

| Step | Key Finding |
|------|------------|
| **EDA** | 208 samples, 60 numeric features, zero missing values. Energy peaks in mid-frequency bands (F20-F45). |
| **PCA** | ~20-25 components retain 95% variance. Clear 2D structure in PC1 vs PC2 scatter. |
| **t-SNE** | Confirms natural groupings visible in PCA space. |
| **K-Means** | Elbow + Silhouette agree on optimal k. Clusters differ in energy profile. |
| **Agglomerative** | Ward linkage produces clusters consistent with K-Means. Dendrogram shows merge hierarchy. |
| **DBSCAN** | Detects tightly-packed regions; flags a few edge points as noise. |
| **GMM** | Probabilistic soft assignments. BIC-optimal components align with K-Means k. |
| **Isolation Forest** | Detects a small number of anomalous signals differing markedly from the majority. |
| **LOF** | Confirms local anomalies; largely agrees with Isolation Forest. |

**Conclusion:** The sonar data contains distinct, learnable structure even without labels.  
K-Means and Agglomerative Clustering consistently recover the same groupings, while anomaly detectors flag a handful of edge-case signals for closer inspection.